In [1]:
!pip install wandb

Сделаю несколько импортов + вандби для логирования

In [2]:
import pandas as pd
import re
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import wandb
wandb.login()
import json
import os

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: artem-1-rubtsov (artem-1-rubtsov-r3) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Заранее обозначу несколько функций, которые далее буду использовать

In [4]:
#предочистка текста от плохих символов
def clean_review(text):
  text = str(text)
  text = re.sub(r"<.*?>", " ", text)
  text = re.sub(r"\s+", " ", text)
  return text.strip()

#токенизация
def tokenize(text):
  text = str(text).lower()
  text = re.sub(r"[^a-zA-Zа-яА-Я0-9]+", " ", text)
  return text.split()

#превращение текста в набор чисел для нейронки
def encode(text, dictionary, maximum):
  tokens = tokenize(text)
  array_index = list(dictionary.get(token, dictionary['unknown']) for token in tokens)
  if len(array_index) < maximum:
    add_buffer = maximum - len(array_index)
    array_index = array_index + [dictionary['add_pad']] * add_buffer
  else:
    array_index = array_index[:maximum]
  return array_index

In [5]:
df = pd.read_csv('data/reviews.csv')
df = df.drop_duplicates()
# создаем новые признаки, с которыми будем работать
df["review_clean"] = df["review"].apply(clean_review)
df["n_words"] = df["review_clean"].apply(lambda x: len(x.split()))
df["target"] = df["sentiment"].apply(lambda x: 1 if x == "positive" else 0)
df.head()

,review,sentiment,review_clean,n_words,target
0,One of the other reviewers has mentioned that ...,positive,One of the other reviewers has mentioned that ...,304,1
1,A wonderful little production. <br /><br />The...,positive,A wonderful little production. The filming tec...,156,1
2,I thought this was a wonderful way to spend ti...,positive,I thought this was a wonderful way to spend ti...,164,1
3,Basically there's a family where a little boy ...,negative,Basically there's a family where a little boy ...,135,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"Petter Mattei's ""Love in the Time of Money"" is...",225,1


In [6]:
train, buffer = train_test_split(df, test_size=0.3, random_state=42, stratify=df["target"])
val, test = train_test_split(buffer, test_size=0.5, random_state=42, stratify=buffer["target"])
train.shape, val.shape, test.shape

((34707, 5), (7437, 5), (7438, 5))

In [7]:
limit = 15000
minimum = 2
array_tokens = []
for item in train['review_clean']:
  tokens = tokenize(item)
  array_tokens.extend(tokens)
s_count = pd.Series(array_tokens).value_counts() # словарь из частот слов

In [8]:
s_count.head()

,count
the,464659
and,225801
a,225473
of,201530
to,186568


In [9]:
token_dict = {} # будем хранить тут слова под своими индексами для нейросети
token_dict['add_pad'] = 0
token_dict['unknown'] = 1
for elem, count in s_count.head(limit).items():
  if count >= minimum:
    token_dict[elem] = len(token_dict)
len(token_dict)

15001

Теперь сохраняем получившиеся выборки и словарь для дальнейшего логирования

In [10]:
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)
with open("token_dict.json", "w") as f:
    json.dump(token_dict, f, ensure_ascii=False)

Создадим класс, с которым будет работать торч

In [11]:
maximum = 300
class ReviewDataset(Dataset):
  def __init__(self, df, dictionary, maximum):
    self.text = df["review_clean"].values
    self.target = df["target"].values
    self.dictionary = dictionary
    self.maximum = maximum

  def __len__(self):
    return len(self.text)

  def __getitem__(self, i):
    text_i = encode(self.text[i], self.dictionary, self.maximum)
    target = self.target[i]
    return {"input": torch.tensor(text_i, dtype=torch.long), "target": torch.tensor(target, dtype=torch.long)}

In [12]:
batch_size = 64
train_dataset = ReviewDataset(train, token_dict, maximum)
val_dataset = ReviewDataset(val, token_dict, maximum)
test_dataset = ReviewDataset(test, token_dict, maximum)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Теперь уже непосредственно к моделям

Первая простая модель

In [14]:
class SimpleClassifier(nn.Module):
  def __init__(self, dict_size, embedding_dim, hidden_dim, num_classes, pad_idx):
    super().__init__()

    self.embedding = nn.Embedding(num_embeddings=dict_size, embedding_dim=embedding_dim, padding_idx=pad_idx)

    self.classifier = nn.Sequential(
        nn.Linear(embedding_dim, hidden_dim),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(hidden_dim, num_classes)
    )

  def forward(self, input):
    emb = self.embedding(input)
    emb = emb * (input != 0).unsqueeze(-1)
    lens = (input != 0).unsqueeze(-1).sum(dim=1).clamp(min=1)
    pool = emb.sum(dim=1) / lens
    output = self.classifier(pool)

    return output

Вторая модель - LSTM

In [15]:
class LSTMClassifier(nn.Module):
  def __init__(self, dict_size, embedding_dim, hidden_dim, num_layers, num_classes, pad_idx):
    super().__init__()

    self.embedding = nn.Embedding(num_embeddings=dict_size, embedding_dim=embedding_dim, padding_idx=pad_idx)

    self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)

    self.classifier = nn.Sequential(
        nn.Linear(hidden_dim * 2, hidden_dim),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(hidden_dim, num_classes)
    )

  def forward(self, input):
    emb = self.embedding(input)
    output, (hidden, cell) = self.lstm(emb)
    hidden_forward = hidden[-2]
    hidden_back = hidden[-1]
    hidden_output = torch.cat((hidden_forward, hidden_back), dim=1)
    output = self.classifier(hidden_output)

    return output

Функция для проверки и оценки качества модели

In [16]:
def eval_model(model, data_loader, criterion, device):
  model.eval()

  total_loss = 0
  array_target = []
  array_pred = []

  with torch.no_grad():
    for batch in data_loader:
      input = batch["input"].to(device)
      target = batch["target"].to(device)
      output = model(input)
      loss = criterion(output, target)
      pred = torch.argmax(output, dim=1)
      total_loss += loss.item()
      array_target.extend(target.cpu().numpy())
      array_pred.extend(pred.cpu().numpy())

  #создаем словарь с информацией по результатам оценки модели
  metric = {"loss": total_loss / len(data_loader), "accuracy": accuracy_score(array_target, array_pred),
            "precision": precision_score(array_target, array_pred), "recall": recall_score(array_target, array_pred),
            "f1": f1_score(array_target, array_pred)}
  return metric

Основная функция для обучения модели

In [17]:
def train_model(model, model_name, train_loader, val_loader, test_loader, criterion, optimizer, device, epochs, params):
  # запускаем эксперимент и логируем все необходимое
  run = wandb.init(project="classifier_reviews", name=model_name, config=params)
  data_artifact = wandb.Artifact("data", type="dataset")
  data_artifact.add_file("train.csv")
  data_artifact.add_file("val.csv")
  data_artifact.add_file("test.csv")
  data_artifact.add_file("token_dict.json")
  wandb.log_artifact(data_artifact)
  for epoch in range(epochs):
    model.train()
    loss_train = 0

    for batch in train_loader:
      input = batch["input"].to(device)
      target = batch["target"].to(device)
      optimizer.zero_grad()
      output = model(input)
      loss = criterion(output, target)
      loss.backward()
      optimizer.step()
      loss_train += loss.item()

    loss_train = loss_train / len(train_loader)
    val_res = eval_model(model, val_loader, criterion, device)
    # логируем результаты на валидации
    wandb.log({"epoch": epoch + 1, "train_loss": loss_train, "val_loss": val_res["loss"],
               "val_accuracy": val_res["accuracy"], "val_precision": val_res["precision"], "val_recall": val_res["recall"], "val_f1": val_res["f1"]})

  test_res = eval_model(model, test_loader, criterion, device)
  # логируем результаты на тесте
  wandb.log({"test_loss": test_res["loss"], "test_accuracy": test_res["accuracy"],
               "test_precision": test_res["precision"], "test_recall": test_res["recall"], "test_f1": test_res["f1"]})

  # сохраняем обученную модель, ее веса
  path = f"{model_name}.pt"
  torch.save(model.state_dict(), path)
  model_artifact = wandb.Artifact(model_name, type="model")
  model_artifact.add_file(path)
  wandb.log_artifact(model_artifact)
  wandb.finish()
  return test_res

Обучаем простую модельку

In [18]:
# параметры модели
dict_size = len(token_dict)
pad_idx = token_dict["add_pad"]
num_classes = 2
embedding_dim = 128
hidden_dim = 128
epochs = 5
lr = 0.001

simple_model = SimpleClassifier(dict_size=dict_size, embedding_dim=embedding_dim,
                                hidden_dim=hidden_dim, num_classes=num_classes, pad_idx=pad_idx).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(simple_model.parameters(), lr=lr)

simple_params = {
    "model_name": "SimpleClassifier",
    "dict_size": dict_size,
    "embedding_dim": embedding_dim,
    "hidden_dim": hidden_dim,
    "num_classes": num_classes,
    "maximum": maximum,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "optimizer": "Adam",
    "loss": "CrossEntropyLoss"
}

# обучение
simple_test_res = train_model(
    model=simple_model,
    model_name="SimpleClassifier",
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    params=simple_params
)

epoch,▁▃▅▆█
test_accuracy,▁
test_f1,▁
test_loss,▁
test_precision,▁
test_recall,▁
train_loss,█▄▃▂▁
val_accuracy,▁▆███
val_f1,▁▆██▇
val_loss,█▂▁▂▅
+2,...


Теперь обучаем LSTM

In [19]:
# параметры модели
dict_size = len(token_dict)
pad_idx = token_dict["add_pad"]
num_classes = 2
embedding_dim = 128
hidden_dim = 128
num_layers = 1
epochs = 5
lr = 0.001

lstm_model = LSTMClassifier(dict_size=dict_size, embedding_dim=embedding_dim,
                            hidden_dim=hidden_dim, num_layers=num_layers, num_classes=num_classes, pad_idx=pad_idx).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=lr)

lstm_params = {
    "model_name": "LSTMClassifier",
    "dict_size": dict_size,
    "embedding_dim": embedding_dim,
    "hidden_dim": hidden_dim,
    "num_layers": num_layers,
    "bidirectional": True,
    "num_classes": num_classes,
    "maximum": maximum,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "optimizer": "Adam",
    "loss": "CrossEntropyLoss"
}

# обучение
lstm_test_res = train_model(
    model=lstm_model,
    model_name="LSTMClassifier",
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    params=lstm_params
)

wandb: WARNING Artifact "data" already exists with the same content. No new version will be created.


epoch,▁▃▅▆█
test_accuracy,▁
test_f1,▁
test_loss,▁
test_precision,▁
test_recall,▁
train_loss,█▅▄▂▁
val_accuracy,▁▄▇▆█
val_f1,▁▅▇▆█
val_loss,█▅▁▃▁
+2,...
